# 🗄️ SQLFixEnv — Live Demo

> **SQL Query Optimizer RL Environment** | Meta × HuggingFace PyTorch Hackathon 2025

This notebook demonstrates all 5 difficulty levels of SQLFixEnv — a reinforcement learning environment where AI agents fix broken SQL queries.

**🔗 Links:**
- HF Space: https://huggingface.co/spaces/SANGRALRAHUL/SQLFixEnv
- GitHub: https://github.com/sangralrahul/SQLFixEnv

---

In [ ]:
# Install dependencies
!pip install requests -q

In [ ]:
import requests
import json

BASE = "https://SANGRALRAHUL-sqlfixenv.hf.space"

def pretty(data):
    print(json.dumps(data, indent=2))

# Health check
resp = requests.get(f"{BASE}/health").json()
print("✅ Environment Status:", resp)

## 📋 All 10 Tasks Across 5 Levels

In [ ]:
tasks = requests.get(f"{BASE}/tasks").json()["tasks"]

for t in tasks:
    print(f"Task {t['id']:2d} | {t['level']:8s} | {t['description'][:60]}...")
    print(f"          Broken: {t['broken_query'][:70]}")
    print()

## 🗃️ Database Schema

In [ ]:
schema = requests.get(f"{BASE}/schema").json()["schema"]

for table, cols in schema.items():
    col_names = ", ".join(c['name'] for c in cols)
    print(f"📊 {table:15s} → {col_names}")

## 🟢 Level 1: Easy — Keyword Typos

In [ ]:
# Reset to task 1
obs = requests.post(f"{BASE}/reset", json={"task_id": 1}).json()
print(f"Level: {obs['level']}")
print(f"Broken Query: {obs['broken_query']}")
print(f"Hint: {obs['hint']}")

In [ ]:
# Agent submits broken query (baseline — should get 0)
obs = requests.post(f"{BASE}/reset", json={"task_id": 1}).json()
result = requests.post(f"{BASE}/step", json={"action": obs["broken_query"]}).json()
print(f"❌ Broken query reward: {result['reward']} | Feedback: {result['feedback']}")

# Agent submits correct query (perfect score)
obs = requests.post(f"{BASE}/reset", json={"task_id": 1}).json()
fixed = "SELECT name, salary FROM employees ORDER BY salary DESC"
result = requests.post(f"{BASE}/step", json={"action": fixed}).json()
print(f"✅ Fixed query reward: {result['reward']} | Feedback: {result['feedback']}")
print(f"   Breakdown: {result['reward_breakdown']}")

## 🟡 Level 2: Medium — Aggregations & HAVING

In [ ]:
obs = requests.post(f"{BASE}/reset", json={"task_id": 4}).json()
print(f"Broken: {obs['broken_query']}")

fixed = "SELECT department_id, AVG(salary) as avg_sal FROM employees GROUP BY department_id HAVING AVG(salary) > 75000 ORDER BY avg_sal DESC"
result = requests.post(f"{BASE}/step", json={"action": fixed}).json()
print(f"✅ Reward: {result['reward']} | Success: {result['success']}")

## 🟠 Level 3: Hard — Multi-Table JOINs

In [ ]:
obs = requests.post(f"{BASE}/reset", json={"task_id": 6}).json()
print(f"Broken: {obs['broken_query']}")

fixed = "SELECT e.name, p.name as project FROM employees e JOIN assignments a ON e.id = a.employee_id JOIN projects p ON a.project_id = p.id"
result = requests.post(f"{BASE}/step", json={"action": fixed}).json()
print(f"✅ Reward: {result['reward']} | Success: {result['success']}")
print(f"   Rows returned: {len(result['agent_result']['rows'])}")

## 🔴 Level 4: Expert — Correlated Subqueries

In [ ]:
obs = requests.post(f"{BASE}/reset", json={"task_id": 7}).json()
print(f"Broken: {obs['broken_query']}")

fixed = "SELECT e.name, e.salary, e.department_id FROM employees e WHERE e.salary > (SELECT AVG(salary) FROM employees WHERE department_id = e.department_id) ORDER BY e.department_id, e.salary DESC"
result = requests.post(f"{BASE}/step", json={"action": fixed}).json()
print(f"✅ Reward: {result['reward']} | Success: {result['success']}")

## ⚫ Level 5: Master — CTEs & Window Functions

In [ ]:
obs = requests.post(f"{BASE}/reset", json={"task_id": 9}).json()
print(f"Broken: {obs['broken_query']}")

fixed = "WITH ranked AS (SELECT name, salary, department_id, RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) as rnk FROM employees) SELECT * FROM ranked WHERE rnk <= 2"
result = requests.post(f"{BASE}/step", json={"action": fixed}).json()
print(f"✅ Reward: {result['reward']} | Success: {result['success']}")
print(f"   Top 2 earners per dept:")
for row in result['agent_result']['rows']:
    print(f"   {row}")

## 🏆 Full Benchmark — All 10 Tasks

In [ ]:
# Perfect agent solutions
solutions = {
    1: "SELECT name, salary FROM employees ORDER BY salary DESC",
    2: "SELECT department_id, COUNT(*) AS total FROM employees GROUP BY department_id",
    3: "SELECT name, salary, hire_date FROM employees WHERE salary > 80000 AND hire_date > '2019-01-01' ORDER BY hire_date",
    4: "SELECT department_id, AVG(salary) as avg_sal FROM employees GROUP BY department_id HAVING AVG(salary) > 75000 ORDER BY avg_sal DESC",
    5: "SELECT e.name, d.name as dept_name FROM employees e INNER JOIN departments d ON e.department_id = d.id ORDER BY d.name",
    6: "SELECT e.name, p.name as project FROM employees e JOIN assignments a ON e.id = a.employee_id JOIN projects p ON a.project_id = p.id",
    7: "SELECT e.name, e.salary, e.department_id FROM employees e WHERE e.salary > (SELECT AVG(salary) FROM employees WHERE department_id = e.department_id) ORDER BY e.department_id, e.salary DESC",
    8: "SELECT d.name, COUNT(p.id) as num_projects, SUM(p.budget) as total_budget FROM departments d LEFT JOIN projects p ON d.id = p.department_id GROUP BY d.id, d.name ORDER BY total_budget DESC",
    9: "WITH ranked AS (SELECT name, salary, department_id, RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) as rnk FROM employees) SELECT * FROM ranked WHERE rnk <= 2",
    10: "SELECT e.name, d.name as department, (s.amount + s.bonus) as total_comp, COUNT(DISTINCT a.project_id) as num_projects FROM employees e JOIN departments d ON e.department_id = d.id JOIN salaries s ON e.id = s.employee_id LEFT JOIN assignments a ON e.id = a.employee_id GROUP BY e.id, e.name, d.name, s.amount, s.bonus ORDER BY total_comp DESC",
}

total = 0.0
print("🏆 SQLFixEnv Perfect Agent Benchmark\n")
print(f"{'Task':5s} {'Level':10s} {'Reward':8s} {'Status'}")
print("-" * 40)

for task_id, solution in solutions.items():
    requests.post(f"{BASE}/reset", json={"task_id": task_id})
    result = requests.post(f"{BASE}/step", json={"action": solution}).json()
    total += result['reward']
    status = "✅ PERFECT" if result['success'] else f"⚠️  {result['reward']}"
    print(f"{task_id:5d} {result['level']:10s} {result['reward']:8.3f} {status}")

print("-" * 40)
print(f"{'TOTAL':5s} {'':10s} {total:8.3f} / 10.0")
print(f"Accuracy: {total/10*100:.1f}%")

## 🎯 Partial Reward Demo
Shows how agents get credit for partially correct solutions

In [ ]:
requests.post(f"{BASE}/reset", json={"task_id": 5})

attempts = [
    ("Wrong query entirely", "SELECT * FROM employees"),
    ("Right table, wrong JOIN", "SELECT e.name, d.name FROM employees e, departments d"),
    ("Almost correct", "SELECT e.name, d.name as dept_name FROM employees e INNER JOIN departments d ON e.department_id = d.id"),
    ("Perfect fix", "SELECT e.name, d.name as dept_name FROM employees e INNER JOIN departments d ON e.department_id = d.id ORDER BY d.name"),
]

print("📈 Partial Reward Progression:\n")
for desc, query in attempts:
    requests.post(f"{BASE}/reset", json={"task_id": 5})
    result = requests.post(f"{BASE}/step", json={"action": query}).json()
    bar = "█" * int(result['reward'] * 20)
    print(f"{desc:35s} | {result['reward']:.2f} |{bar}|")

---
## 🚀 Summary

| Feature | Details |
|---------|--------|
| Difficulty Levels | 5 (easy → master) |
| Total Tasks | 10 |
| Database Tables | 5 (employees, departments, projects, assignments, salaries) |
| Reward Type | Multi-dimensional partial rewards |
| Max Reward | 1.0 per task |
| SQL Features | Keywords, JOINs, Subqueries, CTEs, Window Functions |

**Built for Meta × HuggingFace PyTorch Hackathon 2025**

🔗 https://huggingface.co/spaces/SANGRALRAHUL/SQLFixEnv